# Assignment 09: Array Algorithms (100 points)

## Context

Many USAAIO problems boil down to **array algorithms**: sorting, searching, ranking, cumulative operations, and sliding windows -- all without loops. These are the building blocks of evaluation metrics, data preprocessing, and feature engineering.

The challenge is expressing inherently sequential algorithms (like running max, sliding window, or multi-key sort) as vectorized NumPy operations. The reward: code that is both faster and more readable than explicit loops.

### Key Techniques

- **`np.cumsum` / `np.cumprod`**: running aggregations in $O(N)$
- **`np.argsort`**: returns indices that would sort an array (the foundation of ranking)
- **`np.searchsorted`**: binary search on sorted arrays
- **`np.unique`**: unique values, counts, and inverse mapping
- **Cumsum trick**: compute sliding window sums as `cumsum[i+w] - cumsum[i]`
- **`np.maximum.accumulate`**: running maximum (useful for drawdown calculations)

### Notation

- Shape annotations are required on every intermediate step
- `N` = number of elements, `K` = window size or top-K

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else** for the following purposes:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.
>
> - **ABSOLUTELY NO EXPLICIT LOOPS** (`for`, `while`, list comprehensions) in any solution.
> - These problems are USAAIO contest-style: think vectorized.

---

## Part 1 (15 points, coding task)

**Reasoning is not required.**

Compute running (cumulative) statistics over a 1D array.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
data = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5])

1. **(3 pts)** Running sum: `result[i] = sum of data[0:i+1]`. Store in `running_sum`. Use `np.cumsum`.

2. **(4 pts)** Running maximum: `result[i] = max of data[0:i+1]`. Store in `running_max`. Use `np.maximum.accumulate`.

3. **(4 pts)** Running mean: `result[i] = mean of data[0:i+1]`. Store in `running_mean`. **Hint**: `running_sum / np.arange(1, len(data) + 1)`.

4. **(4 pts)** Running product: `result[i] = product of data[0:i+1]`. Store in `running_product`. Use `np.cumprod`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

running_sum = None
running_max = None
running_mean = None
running_product = None

""" END OF THIS PART """

---

Ranking is a fundamental operation for evaluation metrics. `np.argsort` returns the indices that would sort an array -- applying it twice gives ranks. This double-argsort trick is one of the most useful patterns in competitive programming with NumPy.

---

## Part 2 (20 points, coding task)

**Reasoning is not required.**

Implement ranking and percentile computation without loops.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
scores = np.array([78, 92, 65, 88, 95, 71, 83, 90, 85, 77])

1. **(6 pts)** Rank each score where 1 = highest. Store in `ranks`. For ties, the rank should reflect the position in sorted order (standard competition ranking is fine). **Hint**: `argsort` the negated scores, then `argsort` again and add 1.

2. **(6 pts)** Compute the percentile rank of each score: `percentile[i] = (number of scores <= scores[i]) / total_scores * 100`. Store in `percentiles`. Use broadcasting: `(scores[:, None] >= scores[None, :])` sums to count.

3. **(8 pts)** Find the top K=3 scores and their original indices. Store the K largest values (sorted descending) in `top_k_values` and their original indices in `top_k_indices`. Use `np.argsort` or `np.argpartition`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

K = 3

ranks = None            # (10,) -- 1-based ranks, 1 = highest
percentiles = None      # (10,) -- percentile rank of each score
top_k_values = None     # (3,) -- the K largest values, descending
top_k_indices = None    # (3,) -- their original indices

""" END OF THIS PART """

---

Sorting a matrix by a specific column -- or by multiple keys -- is essential for data analysis. `np.lexsort` sorts by multiple keys but has a non-obvious API: it sorts by the **last** key first (like a stable sort applied in reverse order).

---

## Part 3 (20 points, coding task)

**Reasoning is not required.**

Sort a matrix in various ways without loops.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
M = np.array([
    [3, 1, 4],
    [1, 5, 9],
    [2, 6, 5],
    [3, 5, 8],
    [9, 7, 9]
])  # shape: (5, 3)

1. **(5 pts)** Sort each row independently in ascending order. Store in `sorted_rows`. Expected shape: `(5, 3)`.

2. **(5 pts)** Sort the entire matrix by the **second column** (column index 1) in ascending order. Store in `sorted_by_col1`. **Hint**: use `np.argsort` on column 1, then index `M` with the result.

3. **(5 pts)** Multi-key sort: sort by column 0 ascending, then by column 2 descending for ties in column 0. Store in `sorted_multi`. **Hint**: `np.lexsort` sorts by last key first. Negate column 2 for descending.

4. **(5 pts)** For each row, find the **column index** of the median value. Store in `median_indices` of shape `(5,)`. **Hint**: sort each row, find the median, then use `np.argmin(np.abs(M - median[:, None]), axis=1)`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

sorted_rows = None       # (5, 3)
sorted_by_col1 = None    # (5, 3)
sorted_multi = None      # (5, 3)
median_indices = None     # (5,)

""" END OF THIS PART """

---

Sliding window operations are ubiquitous in time series and signal processing. The **cumsum trick** lets you compute any window sum in $O(N)$ instead of $O(N \times W)$: if `cs = cumsum(data)`, then the sum of elements in window $[i, i+w)$ is `cs[i+w] - cs[i]`.

---

## Part 4 (25 points, coding task)

**Reasoning is not required.**

Compute sliding window statistics and financial metrics without loops.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
prices = np.array([100, 102, 98, 105, 110, 107, 112, 108, 115, 120], dtype=float)
window = 3

1. **(6 pts)** Moving average with window size 3: `result[i] = mean(prices[i:i+3])` for $i = 0, \ldots, 7$. Store in `moving_avg`. Expected length: `len(prices) - window + 1 = 8`. **Use the cumsum trick**: `cs = np.cumsum(prices)`, prepend a 0, then `(cs[w:] - cs[:-w]) / w`.

2. **(6 pts)** Moving maximum with window size 3. Store in `moving_max`. Expected length: 8. **Hint**: use `np.lib.stride_tricks.sliding_window_view(prices, window)` then `.max(axis=1)`.

3. **(6 pts)** Daily returns: `returns[i] = (prices[i+1] - prices[i]) / prices[i]`. Store in `daily_returns`. Expected length: 9.

4. **(7 pts)** Maximum drawdown: the largest peak-to-trough decline. For each point, compute `running_peak - current_price` where `running_peak = np.maximum.accumulate(prices)`. The max drawdown is the largest such value. Store the scalar in `max_drawdown`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

moving_avg = None       # (8,)
moving_max = None       # (8,)
daily_returns = None    # (9,)
max_drawdown = None     # scalar

""" END OF THIS PART """

---

The confusion matrix is the foundation of classification evaluation. Building it without loops requires a clever indexing trick: treat `(true_label, predicted_label)` pairs as 2D indices into the matrix and count occurrences. `np.add.at` or the ravel-bincount trick handles this efficiently.

---

## Part 5 (20 points, coding task)

**Reasoning is not required.**

Implement set operations and a confusion matrix without loops.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
predictions = np.array([0, 1, 2, 1, 0, 2, 1, 0, 2, 1, 0, 0, 1, 2, 2])
ground_truth = np.array([0, 1, 1, 1, 0, 2, 0, 0, 2, 1, 1, 0, 1, 2, 0])
num_classes = 3

1. **(4 pts)** Find the unique classes in `predictions` and count occurrences of each. Store in `unique_classes` and `class_counts`. Use `np.unique` with `return_counts=True`.

2. **(6 pts)** Create a confusion matrix of shape `(num_classes, num_classes)` where `confusion[i, j]` = count of samples where `ground_truth == i` and `predictions == j`. Store in `confusion`. **Hint**: use `np.add.at` on a zero matrix, or `np.bincount` with raveled indices.

3. **(5 pts)** From the confusion matrix, compute per-class accuracy: `accuracy[i] = confusion[i, i] / sum(confusion[i, :])`. Store in `per_class_acc` of shape `(3,)`.

4. **(5 pts)** Compute overall accuracy: `(sum of diagonal) / (sum of all elements)`. Store as a scalar in `overall_acc`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

unique_classes = None    # unique class labels
class_counts = None      # count per class

confusion = None         # (3, 3)
per_class_acc = None     # (3,)
overall_acc = None       # scalar

""" END OF THIS PART """